# Controlled factorial 2×2 — Dev-only Kaggle training

This notebook runs three paired seeds (`2026`, `2126`, `2226`) for the small and expanded weak-label pools. It does not mount, read, or evaluate ViLexNorm Test. Each arm trains to eight epochs without early stopping; Dev-selected horizon-3 and horizon-8 artifacts are exported.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

RUNTIME = Path('/kaggle/input/visolexnorm-offline-runtime')  # update slug if needed
REPO = Path('/kaggle/working/VisolexNorm')
BUNDLE = RUNTIME/'source/visolexnorm.bundle'
assert BUNDLE.is_file(), f'Missing offline source bundle: {BUNDLE}'
assert (RUNTIME/'wheelhouse').is_dir(), 'Missing offline Linux CPython 3.12 wheelhouse.'
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git', 'clone', str(BUNDLE), str(REPO)], check=True)
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
manifest = json.loads((RUNTIME/'offline-runtime-manifest.json').read_text())
assert SOURCE_COMMIT == manifest['source_commit'], (SOURCE_COMMIT, manifest['source_commit'])
print('Offline frozen source revision:', SOURCE_COMMIT)

In [ ]:
%cd {REPO}
sys.path.insert(0, str(REPO))
from visolexnorm.common.offline_runtime import install_runtime
OFFLINE_PACKAGES = Path('/kaggle/working/offline_packages')
runtime_manifest = install_runtime(RUNTIME, OFFLINE_PACKAGES)  # isolated target; pip uses --no-index/--no-cache-dir
os.environ.update({'PIP_NO_INDEX':'1', 'HF_HUB_OFFLINE':'1', 'TRANSFORMERS_OFFLINE':'1'})
import torch, transformers, datasets, sentencepiece, safetensors
assert torch.cuda.is_available(), 'Enable the RTX Pro 6000 GPU accelerator.'
assert sys.version_info[:2] == (3, 12), sys.version
print('Python:', sys.version.split()[0])
print('Runtime:', transformers.__version__, datasets.__version__, sentencepiece.__version__, safetensors.__version__)
print('GPU count:', torch.cuda.device_count(), '| primary:', torch.cuda.get_device_name(0))
print('Offline packages:', OFFLINE_PACKAGES)
print('Offline flags:', os.environ['PIP_NO_INDEX'], os.environ['HF_HUB_OFFLINE'], os.environ['TRANSFORMERS_OFFLINE'])

## Private input dataset
Attach a private Kaggle Dataset as an unpacked directory containing only `checkpoints/model_a/` and the four processed files required below. Do **not** upload a ZIP: unpacking Model A into `/kaggle/working` duplicates several GB and can exhaust disk. Do **not** upload `vilexnorm_test.jsonl` or `outputs/evaluation/`.

In [ ]:
MOUNT = Path('/kaggle/input/visolexnorm-controlled-input')  # update slug if needed
WORK = Path('/kaggle/working/controlled_factorial')
DATA = MOUNT  # Keep data/checkpoint on Kaggle's read-only input mount; never copy or unpack it into WORK.
assert not (MOUNT / 'controlled_training_input.zip').is_file(), 'Attach the Dataset as an unpacked directory, not controlled_training_input.zip.'
required = [
    DATA / 'checkpoints/model_a/config.json',
    DATA / 'data/processed/vilexnorm_train.jsonl',
    DATA / 'data/processed/vilexnorm_dev.jsonl',
    DATA / 'data/processed/visolex_weak_labeled.jsonl',
    DATA / 'data/processed/visolex_weak_labeled_expanded.jsonl',
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, f'Missing input files: {missing}'
assert not (DATA / 'data/processed/vilexnorm_test.jsonl').exists(), 'Do not attach Test.'
assert not (DATA / 'outputs/evaluation').exists(), 'Do not attach historical evaluation outputs.'
def show_disk(label):
    usage = shutil.disk_usage('/kaggle/working')
    print(f'{label}: free={usage.free / 2**30:.2f} GiB, used={usage.used / 2**30:.2f} GiB')

WORK.mkdir(parents=True, exist_ok=True)
show_disk('Before training')
print('Dev-only input inventory verified; input remains on /kaggle/input.')

In [ ]:
# Source provenance is read from the clean Git checkout (REPO); input checksums are read from the private Kaggle Dataset (DATA).
!python -m scripts.controlled_experiments freeze-protocol --source-root {REPO} --data-root {DATA} --config {REPO}/configs/controlled_factorial_config.json --cmax-config {REPO}/configs/c_max20_early_stopping_config.json --output {WORK}/protocol.json

In [ ]:
# Build both paired manifests for each frozen seed.
for seed in (2026, 2126, 2226):
    subprocess.run([
        'python', '-m', 'scripts.controlled_experiments', 'build-factorial',
        '--data-root', str(DATA), '--config', str(REPO/'configs/controlled_factorial_config.json'),
        '--protocol', str(WORK/'protocol.json'), '--seed', str(seed),
        '--output-dir', str(WORK/'runs'),
    ], check=True)

## Smoke gate
Run one 200+200 smoke job. Full runs must use separate work directories and should not use `--smoke-test`.

In [ ]:
SMOKE = WORK / 'smoke_seed_2026_small'
!python -m scripts.controlled_experiments train-factorial --model-a-checkpoint {DATA}/checkpoints/model_a --data-dir {DATA}/data/processed --manifest {WORK}/runs/seed_2026/small/mixture_manifest.json --config {REPO}/configs/controlled_factorial_config.json --work-dir {SMOKE} --smoke-test --no-resume-state
import json
smoke = json.loads((SMOKE/'smoke_test.json').read_text())
assert smoke['passed'] and smoke['composition'] == {'gold': 200, 'pseudo': 200}, smoke
assert smoke['test_inputs_loaded'] is False, smoke
shutil.rmtree(SMOKE)  # Smoke checkpoint is not a scientific artifact and is several GB.
show_disk('After smoke cleanup')
smoke

## Full factorial trajectories — one non-interactive Save Version job
A Kaggle Save Version runs from top to bottom without an opportunity to edit variables. This cell therefore executes all six frozen trajectories in order. It uses disk-safe mode without AdamW resume state; after each successful trajectory it verifies Horizon 3/8 scientific artifacts and removes only the large `best/` checkpoint. If any trajectory fails, the Save Version stops with its `seed/arm` identifier and does not clean that partial run.

In [ ]:
TRAJECTORIES = [(seed, arm) for seed in (2026, 2126, 2226) for arm in ('small', 'expanded')]
for seed, arm in TRAJECTORIES:
    run_id = f'seed_{seed}/{arm}'
    run_dir = WORK / 'runs' / f'seed_{seed}' / arm
    manifest_path = run_dir / 'mixture_manifest.json'
    assert manifest_path.is_file(), f'{run_id}: missing frozen manifest: {manifest_path}'
    assert not (run_dir/'cleanup_report.json').exists(), f'{run_id}: unexpected completed artifact in a fresh Save Version.'
    show_disk(f'Before {run_id}')
    try:
        subprocess.run([
            'python', '-m', 'scripts.controlled_experiments', 'train-factorial',
            '--model-a-checkpoint', str(DATA/'checkpoints/model_a'),
            '--data-dir', str(DATA/'data/processed'),
            '--manifest', str(manifest_path),
            '--config', str(REPO/'configs/controlled_factorial_config.json'),
            '--work-dir', str(run_dir),
            '--no-resume-state',
        ], check=True)
        required = [
            run_dir/'train_config.json',
            *[run_dir/f'horizon_{horizon}'/name for horizon in (3, 8) for name in ('selection.json', 'dev_predictions.jsonl', 'dev_metrics.json', 'terminal_dev_predictions.jsonl', 'terminal_dev_metrics.json')],
        ]
        missing = [str(path) for path in required if not path.is_file()]
        assert not missing, f'{run_id}: completed without required scientific artifacts: {missing}'
        subprocess.run(['python', '-m', 'scripts.controlled_experiments', 'cleanup-factorial-run', '--work-dir', str(run_dir)], check=True)
        assert not (run_dir/'state').exists() and not (run_dir/'best').exists(), f'{run_id}: cleanup did not release large artifacts'
    except Exception as error:
        raise RuntimeError(f'Controlled factorial failed at {run_id}; partial artifacts were retained for audit.') from error
    finally:
        show_disk(f'After {run_id}')

In [ ]:
completed = sorted(path.parent.relative_to(WORK/'runs').as_posix() for path in (WORK/'runs').glob('seed_*/*/cleanup_report.json'))
expected = [f'seed_{seed}/{arm}' for seed in (2026, 2126, 2226) for arm in ('small', 'expanded')]
missing = [name for name in expected if name not in completed]
assert not missing, f'Do not summarize yet. Complete and clean these trajectories first: {missing}'
!python -m scripts.controlled_experiments summarize-factorial --input-root {WORK}/runs --output {WORK}/factorial_summary.json
summary = json.loads((WORK/'factorial_summary.json').read_text())
assert summary['seeds'] == 3 and summary['test_metrics_used'] is False
summary['effects']

In [ ]:
# Every completed trajectory was already cleaned after verification. The archive contains only small reproducibility artifacts.
assert len(list((WORK/'runs').glob('seed_*/*/cleanup_report.json'))) == 6, 'Run all six trajectories before archiving.'
shutil.make_archive('/kaggle/working/controlled_factorial_artifacts', 'zip', WORK, 'runs')
shutil.copy2(WORK/'protocol.json', '/kaggle/working/controlled_factorial_protocol.json')
shutil.copy2(WORK/'factorial_summary.json', '/kaggle/working/controlled_factorial_summary.json')
print('Download controlled_factorial_artifacts.zip, controlled_factorial_protocol.json, and controlled_factorial_summary.json.')